# 1) Analyse descriptive des données

Descritption générale : Dans ce projet on va entrainer notre modèle sur le jeu de données ASVspoof 2019.
Ce data set contient deux types de fichier audio, les PA (Adresses Physiques) et les LA (Adresses Logiques). Nous utiliserons les LA pour notre projet car ce sont des audios générés par IA. Les PA sont des relectures d'enregistrements déjà existant, donc pas utiles pour notre projet.
On va utiliser deux sous-ensembles fournies avec le data set, train pour l'entrainement du modèle et dev pour la validation et l'évaluation.

Description des données : les audios présents dans le jeu de données LA sont de deux types,
spoof (audios synthétiques) et bonafide (audios réel). 
Les données d'entrainement sont labéllées dans le fichier ASVspoof2019.LA.cm.train.trn.txt et celles d'évaluation dans le fichier ASVspoof2019.LA.cm.dev.trl.txt. Ils indiquent pour chaque audio si c'est un enregistrement réél ou synthétique.

Organisation du projet : on a décidé de séparer nos fonctions de notre notebook pour le garder propre, elles sont donc ranger dans un dossier src.  

In [2]:
import sys
from pathlib import Path

SRC_DIR = Path(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_utils import load_asvspoof_protocol, attach_paths_by_utt
import pandas as pd
import matplotlib.pyplot as plt
DATA_ROOT = Path(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\archive\LA")
PROTO_TRAIN = DATA_ROOT / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.train.trn.txt"
PROTO_DEV   = DATA_ROOT / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.dev.trl.txt"
AUDIO_ROOT_TRAIN  = DATA_ROOT / "ASVspoof2019_LA_train" / "flac"
AUDIO_ROOT_DEV   = DATA_ROOT / "ASVspoof2019_LA_dev" / "flac"

proto_train_df = load_asvspoof_protocol(PROTO_TRAIN)
print("Quelques utt du protocole :")
print(proto_train_df["utt"].head().tolist())
print("Nombre de lignes protocole train :", len(proto_train_df))

print("Quelques fichiers audio dans le train :")
flacs = list(AUDIO_ROOT_TRAIN.iterdir())[:10]
print([p.name for p in flacs])
print([p.stem for p in flacs])

train_df = attach_paths_by_utt(load_asvspoof_protocol(PROTO_TRAIN), AUDIO_ROOT_TRAIN)
dev_df   = attach_paths_by_utt(load_asvspoof_protocol(PROTO_DEV), AUDIO_ROOT_DEV)
print(train_df.head())

Quelques utt du protocole :
['-', '-', '-', '-', '-']
Nombre de lignes protocole train : 25380
Quelques fichiers audio dans le train :
['LA_T_1000137.flac', 'LA_T_1000406.flac', 'LA_T_1000648.flac', 'LA_T_1000824.flac', 'LA_T_1001074.flac', 'LA_T_1001114.flac', 'LA_T_1001169.flac', 'LA_T_1001718.flac', 'LA_T_1001871.flac', 'LA_T_1002656.flac']
['LA_T_1000137', 'LA_T_1000406', 'LA_T_1000648', 'LA_T_1000824', 'LA_T_1001074', 'LA_T_1001114', 'LA_T_1001169', 'LA_T_1001718', 'LA_T_1001871', 'LA_T_1002656']
Empty DataFrame
Columns: [utt, is_spoof, path]
Index: []


In [2]:
train_df['is_spoof'].value_counts(normalize=True).plot(kind='bar', title='Répartition train (vraies vs synthétiques)')
plt.xticks([0,1], ['bonafide', 'spoof'], rotation=0)
plt.show()

print("Nombre total d'échantillons :", len(train_df))
print("Fichiers manquants :", train_df['path'].isna().sum())

NameError: name 'train_df' is not defined

# 2) Pré-Traitement

In [3]:
import numpy as np
import librosa

def extract_features(path, sr=16000):
    y, s = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

X_train = np.vstack([extract_features(p) for p in train_df["path"]])
y_train = train_df["is_spoof"].to_numpy()
X_dev   = np.vstack([extract_features(p) for p in dev_df["path"]])
y_dev   = dev_df["is_spoof"].to_numpy()

X_train.shape, X_dev.shape


ModuleNotFoundError: No module named 'librosa'

# 3) Formalisation du problème
Notre problème est un problème de classification binaire supervisée.
En entrée on a l'extrait d'un audio qu'on converti en un vecteur de caractéristiques MFCC.
Et en sortie un label binaire 1 si spoof 0 si non.
On veut un modèle capable de distinguer les spoof des vrais audios.
On évalue notre modèles grâce aux métriques suivantes : 
Accuracy : proportion de bonnes prédictions.
AUC : qualité de la séparation entre vraies et fausses voix.
EER (Equal Error Rate) : point où le taux de fausses alertes est égal au taux d’erreurs de rejet.

# 4) Modèle de base
On a décidé de commencer par une approche classique utilisant un SVM (Support Vector Machine) qui sera entraîné sur les coefficients MFCC moyens et écart-types de chaque échantillon.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

clf = make_pipeline(StandardScaler(), SVC(probability=True, kernel='rbf', C=2.0, gamma='scale'))
clf.fit(X_train, y_train)

y_pred = clf.predict(X_dev)
y_prob = clf.predict_proba(X_dev)[:, 1]

print("Accuracy :", accuracy_score(y_dev, y_pred))
print("AUC :", roc_auc_score(y_dev, y_prob))
print("Confusion matrix :\n", confusion_matrix(y_dev, y_pred))


NameError: name 'X_train' is not defined